In [ ]:
# --- Constants generator for the experiments notebook ---
# Maps a short name -> (node_path, metric) in the one-axis results schema.
# Re-run and paste the output into the constants cell to regenerate.
#   name pattern: <PREFIX><blk>_<TAIL>   (residual: <PREFIX>_<TAIL>)
#   TAILs: AU/AC = .in acts (uncentered/centered), BU/BC = .out acts,
#          GU/GC = .out grads, KF = kfac, GB = gen. Boundaries use AU/AC/GU/GC/GB.
N_BLOCKS = 41
RESID = {"AFN": "after_final_norm", "BFN": "before_final_norm"}
PROJ  = {"U": "up", "D": "down", "G": "gate"}
BOUND = {"AI": "attn.in", "AO": "attn.out", "ARO": "attn.raw_out",
         "MI": "mlp.in",  "MO": "mlp.out",  "MRO": "mlp.raw_out"}

def _proj(p, i, w):
    b = f"blk{i}.mlp.{w}"
    return [(f"{p}{i}_AU", f"{b}.in", "acts_uncentered"), (f"{p}{i}_AC", f"{b}.in", "acts_centered"),
            (f"{p}{i}_BU", f"{b}.out", "acts_uncentered"), (f"{p}{i}_BC", f"{b}.out", "acts_centered"),
            (f"{p}{i}_GU", f"{b}.out", "grads_uncentered"), (f"{p}{i}_GC", f"{b}.out", "grads_centered"),
            (f"{p}{i}_KF", b, "kfac"), (f"{p}{i}_GB", f"{b}.out", "gen")]

def _layer(i):
    up, dn = f"blk{i}.mlp.up.in", f"blk{i}.mlp.down.out"
    return [(f"L{i}_AU", up, "acts_uncentered"), (f"L{i}_AC", up, "acts_centered"),
            (f"L{i}_BU", dn, "acts_uncentered"), (f"L{i}_BC", dn, "acts_centered"),
            (f"L{i}_GU", dn, "grads_uncentered"), (f"L{i}_GC", dn, "grads_centered"),
            (f"L{i}_KF", f"blk{i}.mlp", "projections_kfac"), (f"L{i}_GB", dn, "gen")]

def _resid(p, leaf):
    return [(f"{p}_AU", leaf, "acts_uncentered"), (f"{p}_AC", leaf, "acts_centered"),
            (f"{p}_BU", leaf, "acts_uncentered"), (f"{p}_BC", leaf, "acts_centered"),
            (f"{p}_GU", leaf, "grads_uncentered"), (f"{p}_GC", leaf, "grads_centered"),
            (f"{p}_KF", leaf, "kfac"), (f"{p}_GB", leaf, "gen")]

def _bound(p, i, suf):
    leaf = f"blk{i}.{suf}"
    return [(f"{p}{i}_AU", leaf, "acts_uncentered"), (f"{p}{i}_AC", leaf, "acts_centered"),
            (f"{p}{i}_GU", leaf, "grads_uncentered"), (f"{p}{i}_GC", leaf, "grads_centered"),
            (f"{p}{i}_GB", leaf, "gen")]

rows = []
for p, leaf in RESID.items(): rows += _resid(p, leaf)
for i in range(N_BLOCKS):
    for p, w in PROJ.items(): rows += _proj(p, i, w)
    rows += _layer(i)
    for p, suf in BOUND.items(): rows += _bound(p, i, suf)

code = "\n".join(f"{name} = {node!r}, {metric!r}" for name, node, metric in rows)
print(code)
